# Setup

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import lxml
from tqdm.auto import tqdm

C:\GitHub\rlp-jym-projects\datatalksclub-zoomcamp\zoomcamp-sma-cohort-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Question 1: [IPO] Withdrawn IPOs by Company Type

In [2]:
url = "https://www.iposcoop.com/ipos-recently-filed/"
recent_ipo = pd.read_html(url)
df_base = recent_ipo[0]
df = df_base[df_base['Expected To Trade'] == 'Withdrawn']

In [3]:
df['Company Type'] = np.select(
    [
        df['Company'].str.contains('Technologies', na=False),
        df['Company'].str.contains('Acquisition Corp|Acquisition Corporation|Corp', na=False, regex=True),
        df['Company'].str.contains('Inc|Incorporated', na=False, regex=True),
        df['Company'].str.contains('Group', na=False, regex=True),
        df['Company'].str.contains('Ltd|Limited', na=False, regex=True),
        df['Company'].str.contains('Holdings|Holding', na=False, regex=True),
    ],
    ['Technologies', 'Acquisition Corp', 'Inc.', 'Group', 'Limited', 'Holdings'],
    default='Others',
)
df.info()

<class 'pandas.DataFrame'>
Index: 33 entries, 5 to 484
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   File Date             33 non-null     str    
 1   Company               33 non-null     str    
 2   Symbol                33 non-null     str    
 3   Managers              33 non-null     str    
 4   Shares (millions)     33 non-null     float64
 5   Price Low             27 non-null     str    
 6   Price High            27 non-null     str    
 7   Est $ Vol (millions)  33 non-null     str    
 8   Expected To Trade     33 non-null     str    
 9   SCOOP Rating          33 non-null     str    
 10  Company Type          33 non-null     str    
dtypes: float64(1), str(10)
memory usage: 6.9 KB


In [4]:
df_clean = df.copy()

In [5]:
df_clean['Price Low'] = df['Price Low'].str.replace('$', '', regex=False).astype(float)
df_clean['Price High'] = df['Price High'].str.replace('$', '', regex=False).astype(float)
df_clean['Est $ Vol (millions)'] = df['Est $ Vol (millions)'].str.replace('$', '', regex=False).astype(float)
df_clean['Avg_price'] = df_clean[['Price Low', 'Price High']].mean(axis=1)
df_clean['Shares_offered_value'] = np.where((df_clean['Shares (millions)'] * df_clean['Avg_price']).notna(), df_clean['Shares (millions)'] * df_clean['Avg_price'], df_clean['Est $ Vol (millions)'])
df_clean.info() # homework expects 32 entries

<class 'pandas.DataFrame'>
Index: 33 entries, 5 to 484
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   File Date             33 non-null     str    
 1   Company               33 non-null     str    
 2   Symbol                33 non-null     str    
 3   Managers              33 non-null     str    
 4   Shares (millions)     33 non-null     float64
 5   Price Low             27 non-null     float64
 6   Price High            27 non-null     float64
 7   Est $ Vol (millions)  33 non-null     float64
 8   Expected To Trade     33 non-null     str    
 9   SCOOP Rating          33 non-null     str    
 10  Company Type          33 non-null     str    
 11  Avg_price             27 non-null     float64
 12  Shares_offered_value  33 non-null     float64
dtypes: float64(6), str(7)
memory usage: 6.9 KB


In [6]:
print(f"A1. Total withdraw value → {round(df_clean.groupby('Company Type').Shares_offered_value.sum().sort_values(ascending=False).tolist()[0])}")

A1. Total withdraw value → 500


# Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)

In [7]:
url2 = "https://www.iposcoop.com/2025-pricings/"
recent_ipo = pd.read_html(url2)
df2_base_default = recent_ipo[0]

In [8]:
df2_base_default.info() # both date and return were str

<class 'pandas.DataFrame'>
RangeIndex: 231 entries, 0 to 230
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Company            231 non-null    str    
 1   Symbol             231 non-null    str    
 2   Industry           231 non-null    str    
 3   Offer Date         231 non-null    str    
 4   Shares (millions)  231 non-null    float64
 5   Offer Price        231 non-null    str    
 6   1st Day Close      231 non-null    str    
 7   Current Price      231 non-null    str    
 8   Return             231 non-null    str    
 9   SCOOP Rating       231 non-null    str    
dtypes: float64(1), str(9)
memory usage: 34.4 KB


In [9]:
df2_base = df2_base_default.copy()
df2_base['Offer Date'] = pd.to_datetime(df2_base['Offer Date'])
df2_base['Return'] = (df2_base['Return'].astype(str).str.replace('%', '', regex=False).str.strip().replace({'': None, 'nan': None, 'None': None, '-': None}))
df2_base['Return'] = pd.to_numeric(df2_base['Return'], errors='coerce')
df2 = df2_base[(df2_base['Offer Date'] < '2025-09-01') & (df2_base.Return != 0)]
df2.info() # homework expects 134

<class 'pandas.DataFrame'>
Index: 146 entries, 68 to 230
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Company            146 non-null    str           
 1   Symbol             146 non-null    str           
 2   Industry           146 non-null    str           
 3   Offer Date         146 non-null    datetime64[us]
 4   Shares (millions)  146 non-null    float64       
 5   Offer Price        146 non-null    str           
 6   1st Day Close      146 non-null    str           
 7   Current Price      146 non-null    str           
 8   Return             146 non-null    float64       
 9   SCOOP Rating       146 non-null    str           
dtypes: datetime64[us](1), float64(2), str(7)
memory usage: 20.8 KB


In [10]:
symbols = df2.Symbol.tolist()

stocks_df = []
stocks_df_missing = []
for s in tqdm(symbols):
    price = yf.Ticker(s).history(period='2y')[['Open', 'High', 'Low', 'Close', 'Volume']].reset_index()
    if price.empty:
        stocks_df_missing.append(s)
        continue
    price['Date'] = pd.to_datetime(price['Date'])
    price['Symbol'] = s
    price = price[['Date', 'Symbol', 'Open', 'High', 'Low', 'Close', 'Volume']]
    stocks_df.append(price)

stocks_df = pd.concat(stocks_df, ignore_index=True)

 23%|██▎       | 34/146 [00:07<00:16,  6.85it/s]HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MJID"}}}
$MJID: No data found, symbol may be delisted
 32%|███▏      | 46/146 [00:10<00:23,  4.17it/s]HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CAEP"}}}
$CAEP: No data found, symbol may be delisted
100%|██████████| 146/146 [00:34<00:00,  4.27it/s]


In [11]:
print(f"{len(stocks_df_missing)} missing → {stocks_df_missing}")

13 missing → ['MJID', 'EMPG', 'CAEP', 'PTNM', 'AHL', 'CEPT', 'SDM', 'TBH', 'AGH', 'EPWK', 'MTSR', 'SKBL', 'MCTR']


In [12]:
stocks_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45956 entries, 0 to 45955
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype                          
---  ------  --------------  -----                          
 0   Date    45956 non-null  datetime64[s, America/New_York]
 1   Symbol  45956 non-null  str                            
 2   Open    45956 non-null  float64                        
 3   High    45956 non-null  float64                        
 4   Low     45956 non-null  float64                        
 5   Close   45956 non-null  float64                        
 6   Volume  45956 non-null  int64                          
dtypes: datetime64[s, America/New_York](1), float64(4), int64(1), str(1)
memory usage: 2.6 MB


In [13]:
# as is from homework
stocks_df['growth_252d'] = stocks_df.Close / stocks_df.Close.shift(252)
stocks_df['volatility']  = stocks_df.Close.rolling(30).std() * np.sqrt(252)

risk_free_rate = 0.05
stocks_df['Sharpe'] = (stocks_df['growth_252d'] - risk_free_rate) / stocks_df['volatility']

stocks_df_filter = stocks_df[stocks_df.Date == '2026-09-11']
print(f"A2: Median Sharpe → {stocks_df_filter.Sharpe.median():.4f} (not in choices)")

A2: Median Sharpe → 0.0501 (not in choices)


In [14]:
# with adjustments in formula to match risk free rate (rfr) unit value
stocks_df['change']      = stocks_df.Close / stocks_df.Close.shift(1)   - 1  
stocks_df['growth_252d'] = stocks_df.Close / stocks_df.Close.shift(252) - 1  # '- 1' to match rfr unit value
stocks_df['volatility']  = stocks_df.change.rolling(30).std() * np.sqrt(252) # uses change, not close, to match rfr

risk_free_rate = 0.05 # this is a percent in decimal form
stocks_df['Sharpe'] = (stocks_df['growth_252d'] - risk_free_rate) / stocks_df['volatility']

stocks_df_filter = stocks_df[stocks_df.Date == '2026-09-11']
print(f"A2: Median Sharpe → {stocks_df_filter.Sharpe.median():.2f}. My answer is not in the list so I'm picking the only negative.")

A2: Median Sharpe → -0.38. My answer is not in the list so I'm picking the only negative.


# Question 3: [IPO] 'Fixed Months Holding Strategy'


In [15]:
df3 = stocks_df.sort_values(['Symbol', 'Date']).copy()
days_in_a_month = 21

for m in range(1, 13):
    col_name = f"future_growth_{m}_m"
    df3[col_name] = 100 * (df3.groupby('Symbol').Close.shift(-(days_in_a_month * m)) / df3.Close - 1
    )

In [16]:
df3_ipo = df3.loc[df3.groupby('Symbol')['Date'].idxmin()].reset_index(drop=True)
df3_ipo.info()

<class 'pandas.DataFrame'>
RangeIndex: 133 entries, 0 to 132
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype                          
---  ------              --------------  -----                          
 0   Date                133 non-null    datetime64[s, America/New_York]
 1   Symbol              133 non-null    str                            
 2   Open                133 non-null    float64                        
 3   High                133 non-null    float64                        
 4   Low                 133 non-null    float64                        
 5   Close               133 non-null    float64                        
 6   Volume              133 non-null    int64                          
 7   growth_252d         132 non-null    float64                        
 8   volatility          132 non-null    float64                        
 9   Sharpe              132 non-null    float64                        
 10  change              132 n

In [17]:
print(df3.filter(like='future_growth_').median().sort_values(ascending=False))
# Highest is first month, followed by consistent cummulative decline

future_growth_1_m     -2.830189
future_growth_2_m     -6.293982
future_growth_3_m    -10.140805
future_growth_4_m    -13.610432
future_growth_5_m    -17.900263
future_growth_6_m    -22.295085
future_growth_7_m    -27.323883
future_growth_8_m    -31.344409
future_growth_9_m    -34.555245
future_growth_10_m   -38.705499
future_growth_11_m   -42.476633
future_growth_12_m   -44.864867
dtype: float64


In [18]:
print(df3.filter(like='future_growth_').mean().sort_values(ascending=False))
# Highest is 8 months → 
# When compared to median:
#   the few winners became multibaggers but
#   the majority of issues fail as early as
#   the first month

future_growth_8_m     2745.920364
future_growth_7_m     2688.325351
future_growth_9_m     2442.275755
future_growth_6_m     2375.700212
future_growth_4_m     2371.589974
future_growth_5_m     2204.103907
future_growth_10_m    2162.514361
future_growth_3_m     2012.101396
future_growth_2_m     1520.357204
future_growth_11_m    1273.958433
future_growth_12_m    1053.177982
future_growth_1_m      885.501900
dtype: float64


In [19]:
print('A3: Based only on median → the "best" time to hold is 1 month')

A3: Based only on median → the "best" time to hold is 1 month


# Question 4: [Strategy] Simple RSI-Based Trading Strategy

In [20]:
# import gdown
# file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
# gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
# df = pd.read_parquet("data.parquet", engine="pyarrow") 

# pd.read_parquet keeps giving an error

In [21]:
import duckdb
df4_base = duckdb.sql("""select * from read_parquet("data.parquet")""").df()
df4_base.info()

<class 'pandas.DataFrame'>
RangeIndex: 229932 entries, 0 to 229931
Columns: 204 entries, Open to __index_level_0__
dtypes: datetime64[ns](3), float64(129), int32(64), int64(6), str(2)
memory usage: 303.4 MB


In [22]:
print(df4_base.columns.tolist())

['Open', 'High', 'Low', 'Close_x', 'Volume', 'Dividends', 'Stock Splits', 'Ticker', 'Year', 'Month', 'Weekday', 'Date', 'growth_1d', 'growth_3d', 'growth_7d', 'growth_30d', 'growth_90d', 'growth_365d', 'growth_future_30d', 'SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative', 'volatility', 'is_positive_growth_30d_future', 'ticker_type', 'index_x', 'adx', 'adxr', 'apo', 'aroon_1', 'aroon_2', 'aroonosc', 'bop', 'cci', 'cmo', 'dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext', 'macdsignal_ext', 'macdhist_ext', 'macd_fix', 'macdsignal_fix', 'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm', 'ppo', 'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk', 'fastd', 'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr', 'index_y', 'ad', 'adosc', 'obv', 'atr', 'natr', 'ht_dcperiod', 'ht_dcphase', 'ht_phasor_inphase', 'ht_phasor_quadrature', 'ht_sine_sine', 'ht_sine_leadsine', 'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wclprice', 'index', 'cdl2crows', '

In [24]:
df4 = df4_base[['Date', 'growth_future_30d', 'rsi']][(df4_base.Date >= '2000-01-01') & (df4_base.Date <= '2025-06-01')]
level = 30
df4['RSI_Oversold']   = np.where((df4.rsi < level), 'oversold', '')
df4['RSI_Crossunder'] = np.where((df4.rsi < level) & (df4.rsi.shift(1) > level), 'oversold', '')
oversold   = df4[df4.RSI_Oversold   != '']
crossunder = df4[df4.RSI_Crossunder != '']
return_oversold   = 1000 * (oversold.growth_future_30d   - 1).sum()
return_crossunder = 1000 * (crossunder.growth_future_30d - 1).sum()
print(f"Net income (oversold = redundant) → {return_oversold/1e3:,.2f}")
print(f"Net income (crossunder = 1 trade) → {return_crossunder/1e3:,.2f}")
# My personal answer is crossunder to avoid redundant trades (average down until stock no longer is oversold).

Net income (oversold = redundant) → 65.81
Net income (crossunder = 1 trade) → 24.02
